#### Simple Gen AI APP Using Langchain

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [4]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

C:\Users\SHUVEL\AppData\Local\Temp\ipykernel_1892\2433114369.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [34]:
# loader=WebBaseLoader("https://docs.smith.langchain.com/tutorials/Administrators/manage_spend")
loader=WebBaseLoader("https://giftem.co/")
loader

In [36]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://giftem.co/', 'title': 'GiFTEM | Recruiting Intelligence Infrastructure', 'description': 'GiFTEM is a role-aware recruiting intelligence layer that replaces fragmented tools with consistent job understanding, explainable candidate evaluation, and context-aware outreach.', 'language': 'en'}, page_content="GiFTEM | Recruiting Intelligence InfrastructureMenuLog In→Log InMenuOver 800 Million Candidate ProfilesRecruit with intelligence.GiFTEM is a role-aware recruiting intelligence layer that replaces fragmented tools with consistent job understanding, explainable candidate evaluation, and context-aware outreach.Book a Demo→See How It Works→app.giftem.io/dashboardCandidatesJobsAnalyticsOutreachSemantic SearchExploreStrictBooleanFiltersExplore Mode(21 results)Showing candidates with flexible semantic matchingJob Title (must)Aliases (nice)Location (must)Experience (must)MMeredith NicholsScientific Programmer Analyst IIScience Systems and Applications, Inc

In [37]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [38]:
documents

[Document(metadata={'source': 'https://giftem.co/', 'title': 'GiFTEM | Recruiting Intelligence Infrastructure', 'description': 'GiFTEM is a role-aware recruiting intelligence layer that replaces fragmented tools with consistent job understanding, explainable candidate evaluation, and context-aware outreach.', 'language': 'en'}, page_content='GiFTEM | Recruiting Intelligence InfrastructureMenuLog In→Log InMenuOver 800 Million Candidate ProfilesRecruit with intelligence.GiFTEM is a role-aware recruiting intelligence layer that replaces fragmented tools with consistent job understanding, explainable candidate evaluation, and context-aware outreach.Book a Demo→See How It Works→app.giftem.io/dashboardCandidatesJobsAnalyticsOutreachSemantic SearchExploreStrictBooleanFiltersExplore Mode(21 results)Showing candidates with flexible semantic matchingJob Title (must)Aliases (nice)Location (must)Experience (must)MMeredith NicholsScientific Programmer Analyst IIScience Systems and Applications, Inc

EMBEDDING MODEL DEFINATION

In [39]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",
)

In [57]:
#Store the embeddings in a vector database
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [41]:
vectorstoredb

In [42]:
## Query From a vector db
query="What is mean Giftem"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'with context across voice, email, and SMSBook a Demo →See How It Works →See how GiFTEM works across your real roles and searches.Built for recruiters who need accuracy, speed, and credibility — not another AI toy.GiFTEM is recruiting intelligence software designed to help recruiters make better decisions, faster, without losing control or credibility.ProductFeaturesVoice AIPricingTestimonialsSolutionsRCMGovTechNurse RecruitingConstructionResourcesManifestoBlogPodcastsFAQsTrust CenterBook a Demo© 2026 GiFTEM. All rights reserved.'

In [ ]:
# Initialize the LLM Model
from langchain_ollama import ChatOllama
llm=ChatOllama(model="Gemma3")

In [ ]:
## Retrieval Chain, Document chain

from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)
# create_stuff_documents_chain is used to create a chain that combines multiple documents into a single document, which can then be processed by a language model.
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, model='Gemma3')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [45]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"What is mean By Giftem?",
    "context":[Document(page_content='with context across voice, email, and SMSBook a Demo →See How It Works →See how GiFTEM works across your real roles and searches.Built for recruiters who need accuracy, speed, and credibility — not another AI toy.GiFTEM is recruiting intelligence software designed to help recruiters make better decisions, faster, without losing control or credibility.ProductFeaturesVoice AIPricingTestimonialsSolutionsRCMGovTechNurse RecruitingConstructionResourcesManifestoBlogPodcastsFAQsTrust CenterBook a Demo© 2026 GiFTEM. All rights reserved.'
)]
    # "context":[Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph. ")]
})

'GiFTEM is recruiting intelligence software designed to help recruiters make better decisions, faster, without losing control or credibility.'

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [46]:
### Input--->Retriever--->vectorstoredb

vectorstoredb

In [ ]:
retriever=vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
# create_retrieval_chain is used to create a chain that retrieves relevant documents from a vector store based on a query, and then processes those documents using a document chain.
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [49]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002541CE53390>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
            | 

In [56]:
## Get the response form the LLM
# response=retrieval_chain.invoke({"input":"What is mean By Giftem?"})
# response['answer']

from IPython.display import Markdown, display

response = retrieval_chain.invoke({"input": "What is mean by GiFTEM?"})
display(Markdown(response["answer"]))

Here's what we can gather from the provided text about GiFTEM:

*   **What it is:** GiFTEM is recruiting intelligence software designed to help recruiters make better decisions faster. It’s built for accuracy, speed, and credibility, not just an AI toy.
*   **Key Features:**
    *   Role-aware recruiting intelligence layer
    *   Consistent job understanding
    *   Explainable candidate evaluation
    *   Context-aware outreach
    *   Automatic extraction of role taxonomy and skill ontology
    *   Semantic Search
*   **How it works:** It analyzes job descriptions to extract structured data, classifies roles into taxonomies, maps skills based on relevance, and provides match insights.
*   **Benefits:** Reduces time-to-hire, improves candidate fit, provides transparent evaluation, and offers context across voice, email, and SMS.
*   **Data:** Over 800 million candidate profiles are available.
*   **Testimonials:** Users praise the resume analyzer and significant reductions in time-to-hire.

In [54]:

response

{'input': 'What is mean By Giftem?',
 'context': [Document(id='ca846ede-8ae1-48c4-8d71-24103a28f266', metadata={'source': 'https://giftem.co/', 'title': 'GiFTEM | Recruiting Intelligence Infrastructure', 'description': 'GiFTEM is a role-aware recruiting intelligence layer that replaces fragmented tools with consistent job understanding, explainable candidate evaluation, and context-aware outreach.', 'language': 'en'}, page_content='with context across voice, email, and SMSBook a Demo →See How It Works →See how GiFTEM works across your real roles and searches.Built for recruiters who need accuracy, speed, and credibility — not another AI toy.GiFTEM is recruiting intelligence software designed to help recruiters make better decisions, faster, without losing control or credibility.ProductFeaturesVoice AIPricingTestimonialsSolutionsRCMGovTechNurse RecruitingConstructionResourcesManifestoBlogPodcastsFAQsTrust CenterBook a Demo© 2026 GiFTEM. All rights reserved.'),
  Document(id='00c3d71d-b6

In [55]:
response['context']

[Document(id='ca846ede-8ae1-48c4-8d71-24103a28f266', metadata={'source': 'https://giftem.co/', 'title': 'GiFTEM | Recruiting Intelligence Infrastructure', 'description': 'GiFTEM is a role-aware recruiting intelligence layer that replaces fragmented tools with consistent job understanding, explainable candidate evaluation, and context-aware outreach.', 'language': 'en'}, page_content='with context across voice, email, and SMSBook a Demo →See How It Works →See how GiFTEM works across your real roles and searches.Built for recruiters who need accuracy, speed, and credibility — not another AI toy.GiFTEM is recruiting intelligence software designed to help recruiters make better decisions, faster, without losing control or credibility.ProductFeaturesVoice AIPricingTestimonialsSolutionsRCMGovTechNurse RecruitingConstructionResourcesManifestoBlogPodcastsFAQsTrust CenterBook a Demo© 2026 GiFTEM. All rights reserved.'),
 Document(id='00c3d71d-b646-404b-b840-c3f1e290c4e9', metadata={'source': 'h